# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

#%pip install docling markdown-it-py
#%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Documents
import glob
from pathlib import Path

# The parser step above produces markdown files in data_dir
list_md_files = sorted(glob.glob(f"{data_dir}/*.md"))

if not list_md_files:
    raise FileNotFoundError(f"No markdown files found in {data_dir}. Run Step 1 first.")

print(f"Found {len(list_md_files)} markdown files")
for fp in list_md_files:
    print(f"  - {Path(fp).name}")


In [ ]:
# Step 4: Text Chunking and Automatic ICL Generation via SDG Hub Flow

from markdown_it import MarkdownIt
from typing import List, Optional
from pathlib import Path
from dotenv import load_dotenv
from sdg_hub import Flow, FlowRegistry
import datasets
import pandas as pd
import json
import os
import re


load_dotenv()


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """Split markdown text into word chunks while preserving some overlap."""
    md = MarkdownIt()
    tokens = md.parse(text)

    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def set_model_config(flow_object):
    """Configure flow model settings from .env, matching knowledge_generation logic."""
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")
    print(f"Using model provider: {model_provider}")

    if model_provider == "hosted_vllm":
        vllm_model = os.getenv("VLLM_MODEL", "hosted_vllm/meta-llama/Llama-3.3-70B-Instruct")
        vllm_api_base = os.getenv("VLLM_API_BASE", "http://localhost:8000/v1")
        vllm_api_key = os.getenv("VLLM_API_KEY", "EMPTY")
        enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in ("1", "true", "yes")
        flow_object.set_model_config(
            model=vllm_model,
            api_base=vllm_api_base,
            api_key=vllm_api_key,
            enable_reasoning=enable_reasoning,
        )
    elif model_provider == "openai":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
        )
    elif model_provider == "openrouter":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
            api_base="https://openrouter.ai/api/v1",
        )
    elif model_provider == "ollama":
        flow_object.set_model_config(
            model=os.getenv("OLLAMA_MODEL", "ollama/gemma2"),
            api_base=os.getenv("OLLAMA_API_BASE", "http://localhost:11434"),
        )
    elif model_provider == "maas":
        flow_object.set_model_config(
            model=os.getenv("MAAS_MODEL"),
            api_base=os.getenv("MAAS_API_BASE"),
            api_key=os.getenv("MAAS_API_KEY"),
        )
    else:
        raise ValueError(f"Unsupported MODEL_PROVIDER: {model_provider}")

    return flow_object


def truncate_words(text: str, max_words: int) -> str:
    words = str(text or "").split()
    if len(words) <= max_words:
        return str(text or "")
    return " ".join(words[:max_words])


def compact_document_for_prompt(text: str, max_words: int = 4500) -> str:
    words = str(text or "").split()
    if len(words) <= max_words:
        return str(text or "")

    head_size = int(max_words * 0.75)
    tail_size = max_words - head_size
    head = " ".join(words[:head_size])
    tail = " ".join(words[-tail_size:])
    return f"{head}\n\n[... CONTENT TRUNCATED ...]\n\n{tail}"


def normalize_query(q: str) -> str:
    cleaned = str(q or "").replace("<Insert question here>", "").strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned


def parse_questions(text: str) -> List[str]:
    if not text:
        return []

    tag_questions = [
        q.strip()
        for q in re.findall(r"\[QUESTION\](.*?)\[END\]", text, flags=re.DOTALL | re.IGNORECASE)
        if q and q.strip()
    ]
    if tag_questions:
        return tag_questions

    # Secondary fallback: question-like lines
    line_candidates = []
    for line in text.splitlines():
        cleaned = line.strip(" -•\t")
        if cleaned.count("?") >= 1 and len(cleaned.split()) >= 5:
            line_candidates.append(cleaned)
    return line_candidates


def fallback_queries(document_outline: str) -> List[str]:
    return [
        f"What are the main ideas presented in {document_outline}?",
        f"How can the key points in {document_outline} be applied in practice?",
        f"What limitations or trade-offs are discussed in {document_outline}?",
    ]


def parse_tag_field(text: str, tag: str) -> str:
    pattern = rf"\[{re.escape(tag)}\](.*?)\[END\]"
    m = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else ""


def parse_json_payload(text: str):
    if not text:
        return None
    try:
        payload = json.loads(text)
        return payload if isinstance(payload, dict) else None
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        payload = json.loads(match.group(0))
        return payload if isinstance(payload, dict) else None
    except json.JSONDecodeError:
        return None


def response_to_text(raw_output) -> str:
    candidates = []
    if isinstance(raw_output, list):
        candidates = [x for x in raw_output if isinstance(x, dict)]
    elif isinstance(raw_output, dict):
        candidates = [raw_output]

    for obj in candidates:
        content = str(obj.get("content") or "").strip()
        if content:
            return content
        reasoning = str(obj.get("reasoning_content") or "").strip()
        if reasoning:
            return reasoning
    return ""


def parse_base_fields_from_text(text: str, doc_name: str, source_document: str) -> dict:
    payload = parse_json_payload(text)

    if payload:
        document_outline = str(payload.get("document_outline") or payload.get("outline") or "").strip()
        domain = str(payload.get("domain") or "").strip()
        icl_document = str(payload.get("icl_document") or payload.get("context") or "").strip()
        seed_query_1 = normalize_query(payload.get("seed_query_1") or payload.get("icl_query_1") or "")
        seed_query_2 = normalize_query(payload.get("seed_query_2") or payload.get("icl_query_2") or "")
        seed_query_3 = normalize_query(payload.get("seed_query_3") or payload.get("icl_query_3") or "")
    else:
        document_outline = parse_tag_field(text, "DOCUMENT_OUTLINE")
        domain = parse_tag_field(text, "DOMAIN")
        icl_document = parse_tag_field(text, "ICL_DOCUMENT")
        seed_query_1 = normalize_query(parse_tag_field(text, "SEED_QUERY_1"))
        seed_query_2 = normalize_query(parse_tag_field(text, "SEED_QUERY_2"))
        seed_query_3 = normalize_query(parse_tag_field(text, "SEED_QUERY_3"))

    outline_default = Path(doc_name).stem.replace("_", " ").strip() or "Document"
    document_outline = document_outline or outline_default
    domain = domain or "articles/essays"
    icl_document = truncate_words(icl_document or source_document, 1600)

    seeds = [seed_query_1, seed_query_2, seed_query_3]
    seed_fallback = fallback_queries(document_outline)
    for i in range(3):
        if not seeds[i]:
            seeds[i] = seed_fallback[i]

    return {
        "document_outline": document_outline,
        "domain": domain,
        "icl_document": icl_document,
        "seed_query_1": seeds[0],
        "seed_query_2": seeds[1],
        "seed_query_3": seeds[2],
    }


def apply_icl_bootstrap_fallback(flow_object):
    """Patch flow blocks to fallback from content -> reasoning_content and avoid empty parser outputs."""
    blocks_by_name = {b.block_name: b for b in flow_object.blocks}

    generate_icl_base = blocks_by_name.get("generate_icl_base")
    generate_icl_questions = blocks_by_name.get("generate_icl_questions")
    extract_icl_base = blocks_by_name.get("extract_icl_base")
    parse_icl_base = blocks_by_name.get("parse_icl_base")
    extract_questions = blocks_by_name.get("extract_questions")
    parse_question_list = blocks_by_name.get("parse_question_list")

    # Jupyter already runs an event loop, so async_mode=True raises runtime errors.
    for llm_block in [generate_icl_base, generate_icl_questions]:
        if llm_block is not None and getattr(llm_block, "async_mode", False):
            llm_block.async_mode = False

    if extract_icl_base is not None:
        extract_icl_base.extract_content = True
        extract_icl_base.extract_reasoning_content = True

        if not getattr(extract_icl_base, "_content_fallback_patched", False):
            original_generate = extract_icl_base.generate

            def _extract_icl_base_with_fallback(samples, **kwargs):
                df = original_generate(samples, **kwargs)
                if df.empty:
                    return df
                content_col = "extract_icl_base_content"
                reasoning_col = "extract_icl_base_reasoning_content"
                output_col = "icl_base_text_for_parsing"
                if content_col not in df.columns:
                    df[content_col] = ""
                if reasoning_col not in df.columns:
                    df[reasoning_col] = ""
                content = df[content_col].fillna("").astype(str).str.strip()
                reasoning = df[reasoning_col].fillna("").astype(str).str.strip()
                df[output_col] = content.where(content != "", reasoning)
                return df

            extract_icl_base.generate = _extract_icl_base_with_fallback
            extract_icl_base._content_fallback_patched = True

    if parse_icl_base is not None:
        parse_icl_base.input_cols = ["icl_base_text_for_parsing"]

        if not getattr(parse_icl_base, "_robust_parse_patched", False):
            original_generate = parse_icl_base.generate

            def _parse_icl_base_with_fallback(samples, **kwargs):
                out = original_generate(samples, **kwargs)
                if not out.empty:
                    return out

                rows = []
                for sample in samples.to_dict("records"):
                    parsed = parse_base_fields_from_text(
                        str(sample.get("icl_base_text_for_parsing") or ""),
                        str(sample.get("document_title") or "document"),
                        str(sample.get("document") or ""),
                    )
                    rows.append({**sample, **parsed})
                return pd.DataFrame(rows)

            parse_icl_base.generate = _parse_icl_base_with_fallback
            parse_icl_base._robust_parse_patched = True

    if extract_questions is not None:
        extract_questions.extract_content = True
        extract_questions.extract_reasoning_content = True

        if not getattr(extract_questions, "_content_fallback_patched", False):
            original_generate = extract_questions.generate

            def _extract_questions_with_fallback(samples, **kwargs):
                df = original_generate(samples, **kwargs)
                if df.empty:
                    return df
                content_col = "extract_questions_content"
                reasoning_col = "extract_questions_reasoning_content"
                output_col = "question_text_for_parsing"
                if content_col not in df.columns:
                    df[content_col] = ""
                if reasoning_col not in df.columns:
                    df[reasoning_col] = ""
                content = df[content_col].fillna("").astype(str).str.strip()
                reasoning = df[reasoning_col].fillna("").astype(str).str.strip()
                df[output_col] = content.where(content != "", reasoning)
                return df

            extract_questions.generate = _extract_questions_with_fallback
            extract_questions._content_fallback_patched = True

    if parse_question_list is not None:
        parse_question_list.input_cols = ["question_text_for_parsing"]

        if not getattr(parse_question_list, "_robust_parse_patched", False):
            original_generate = parse_question_list.generate

            def _parse_question_list_with_fallback(samples, **kwargs):
                out = original_generate(samples, **kwargs)
                if not out.empty:
                    return out

                rows = []
                for sample in samples.to_dict("records"):
                    text = str(sample.get("question_text_for_parsing") or "")
                    questions = [normalize_query(q) for q in parse_questions(text)]
                    questions = [q for q in questions if q]

                    if not questions:
                        questions = [
                            normalize_query(sample.get("seed_query_1", "")),
                            normalize_query(sample.get("seed_query_2", "")),
                            normalize_query(sample.get("seed_query_3", "")),
                        ]
                        questions = [q for q in questions if q]

                    if not questions:
                        questions = fallback_queries(str(sample.get("document_outline") or "Document"))

                    for q in questions:
                        rows.append({**sample, "question": q})

                return pd.DataFrame(rows)

            parse_question_list.generate = _parse_question_list_with_fallback
            parse_question_list._robust_parse_patched = True

    return flow_object


# Load official flow and apply model config/fallbacks
flow_id = "brisk-cinder-icl-bootstrap"
flow_name = "ICL Bootstrap Metadata and Question Seeding Flow"


def register_local_flows_path() -> Optional[str]:
    """Register local source-tree flows so notebooks work without package reinstall."""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        local_flows = base / "src" / "sdg_hub" / "flows"
        if local_flows.exists():
            FlowRegistry.register_search_path(str(local_flows))
            # FlowRegistry caches discovered entries; force refresh after adding a path.
            if hasattr(FlowRegistry, "_discover_flows"):
                FlowRegistry._discover_flows(force_refresh=True)
            return str(local_flows)
    return None


local_flows_path = register_local_flows_path()
if local_flows_path:
    print(f"Registered local flows path: {local_flows_path}")

# Prefer lookup by id, fallback to name for compatibility.
flow_path = FlowRegistry.get_flow_path(flow_id) or FlowRegistry.get_flow_path(flow_name)
if flow_path is None:
    available = []
    try:
        available = FlowRegistry.list_flows()
    except Exception:
        pass
    available_preview = [f"{f.get('id')} ({f.get('name')})" for f in available[:10]]
    raise FileNotFoundError(
        "Flow not found in registry: "
        f"{flow_id} / {flow_name}. "
        f"Available flows (first 10): {available_preview}"
    )

print(f"Using flow id: {flow_id}")
print(f"Using flow name: {flow_name}")
print(f"Flow path: {flow_path}")

icl_bootstrap_flow = Flow.from_yaml(flow_path)
icl_bootstrap_flow = set_model_config(icl_bootstrap_flow)
icl_bootstrap_flow = apply_icl_bootstrap_fallback(icl_bootstrap_flow)


# Process every markdown document and replicate per-doc ICL across its chunks
seed_rows = []
processed_docs = 0
skipped_docs = 0

for md_fp in list_md_files:
    doc_name = Path(md_fp).name
    print(f"\nProcessing: {doc_name}")

    try:
        with open(md_fp, "r", encoding="utf-8") as f:
            doc_text = f.read()

        chunks = chunk_markdown(doc_text, max_tokens=5000, overlap=1000)
        if not chunks:
            print("  - Skipping: no chunks generated")
            skipped_docs += 1
            continue

        flow_input = pd.DataFrame(
            [
                {
                    "document": compact_document_for_prompt(doc_text, max_words=4500),
                    "document_title": Path(md_fp).stem,
                }
            ]
        )

        flow_output = icl_bootstrap_flow.generate(flow_input, max_concurrency=1)
        if flow_output.empty:
            print("  - Skipping: flow output is empty")
            skipped_docs += 1
            continue

        first_row = flow_output.iloc[0].to_dict()
        document_outline = str(first_row.get("document_outline") or "").strip() or Path(md_fp).stem.replace("_", " ")
        domain = str(first_row.get("domain") or "").strip() or "articles/essays"
        icl_document = truncate_words(str(first_row.get("icl_document") or "").strip() or doc_text, 1600)

        seed_qs = [
            normalize_query(first_row.get("seed_query_1", "")),
            normalize_query(first_row.get("seed_query_2", "")),
            normalize_query(first_row.get("seed_query_3", "")),
        ]
        seed_defaults = fallback_queries(document_outline)
        for i in range(3):
            if not seed_qs[i]:
                seed_qs[i] = seed_defaults[i]

        generated_questions = []
        if "question" in flow_output.columns:
            generated_questions = [normalize_query(q) for q in flow_output["question"].tolist()]
            generated_questions = [q for q in generated_questions if q]

        final_questions = []
        for q in generated_questions + seed_qs:
            if q and q not in final_questions:
                final_questions.append(q)

        while len(final_questions) < 3:
            final_questions.append(seed_defaults[len(final_questions)])
        final_questions = final_questions[:3]

        for chunk in chunks:
            seed_rows.append(
                {
                    "document": chunk,
                    "document_outline": document_outline,
                    "icl_document": icl_document,
                    "icl_query_1": final_questions[0],
                    "icl_query_2": final_questions[1],
                    "icl_query_3": final_questions[2],
                    "domain": domain,
                }
            )

        print(f"  - Chunks: {len(chunks)}")
        print(f"  - Outline: {document_outline}")
        print(f"  - Domain: {domain}")
        processed_docs += 1

    except Exception as exc:
        print(f"  - Skipping due to error: {exc}")
        skipped_docs += 1

if not seed_rows:
    raise ValueError("No seed rows generated. Check data_dir and model configuration.")

# Build and validate final dataset schema
required_cols = [
    "document",
    "document_outline",
    "icl_document",
    "icl_query_1",
    "icl_query_2",
    "icl_query_3",
    "domain",
]

seed_data = datasets.Dataset.from_list(seed_rows).select_columns(required_cols)

for col in ["document_outline", "icl_document", "icl_query_1", "icl_query_2", "icl_query_3", "domain"]:
    empty_count = sum(1 for v in seed_data[col] if not str(v).strip())
    if empty_count > 0:
        raise ValueError(f"Column {col} has {empty_count} empty values after ICL generation")

# Save the seed data to a JSONL file for downstream use
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)

print("\n" + "=" * 60)
print("ICL AUTOMATIC GENERATION SUMMARY")
print("=" * 60)
print(f"Processed documents: {processed_docs}")
print(f"Skipped documents:   {skipped_docs}")
print(f"Total output rows:   {len(seed_data)}")
print("Saved: seed_data.jsonl")


### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook